# Python JSON Exercises: 9 Coding Problems with Solutions

A practice notebook on the `json` module — encoding, decoding, pretty-printing, nested access, custom object encoding/decoding, and validation — each with a concept note, a hint, a solution, and an explanation.

*Adapted for practice from the exercise list at [PYnative](https://pynative.com/python-json-exercise/). Exercise 3's source data had a minor inconsistency (2 keys given, 3 shown in the expected output) which is fixed here with self-consistent 3-key data.*

---

## Concepts you'll need

This set covers Python's built-in **`json`** module — converting between Python objects and JSON text.

- **The core conversion pair** — `json.dumps(obj)` serializes a Python object *to* a JSON string ("dump string"); `json.loads(json_string)` parses a JSON string *back into* Python objects ("load string"). The file-based equivalents, `json.dump(obj, file)` and `json.load(file)`, work the same way but read/write directly to a file object instead of returning/accepting a string.
- **Type mapping** — a JSON object `{}` becomes a Python `dict`; a JSON array `[]` becomes a `list`; JSON strings/numbers/booleans/`null` map to `str`/`int` or `float`/`bool`/`None`.
- **Formatting output** — `indent=N` pretty-prints with N-space indentation (vs. `dumps()`'s default compact single-line output); `sort_keys=True` alphabetizes keys; `separators=(item_sep, key_sep)` controls the exact punctuation between items and between a key and its value.
- **Accessing nested data** — once parsed into nested dicts/lists, reach any depth by chaining `[...]` lookups: `data['company']['employee']['payable']['salary']`.
- **Encoding custom objects** — `json.dumps()` only knows how to serialize built-in types by default, and raises `TypeError` on a custom class instance. Two ways around this: pass `obj.__dict__` directly (works when you just want the instance's plain attributes), or subclass `JSONEncoder` and override its `.default()` method for more control, passed via `cls=YourEncoder`.
- **Decoding into custom objects** — `json.loads(text, object_hook=your_function)` calls `your_function` on every JSON object as it's parsed, letting you convert plain dicts into instances of your own classes on the fly.
- **Validating JSON** — attempting `json.loads()` on malformed text raises `json.JSONDecodeError` (a subclass of the more general `ValueError`), which you can catch to check validity without crashing.
- **Extracting values from a list of objects** — once parsed, a JSON array of objects becomes a Python list of dicts; a list comprehension like `[item['name'] for item in data]` pulls out just one field from every entry.

Each exercise below gives a problem, a hint, a solution, and an explanation.

## Exercise 1. Convert a Dictionary Into JSON

**Concept:** json.dumps()

**Problem:** Convert a Python dictionary into a JSON-formatted string.

**Given:**
```
data = {"key1": "value1", "key2": "value2"}
```

**Expected Output:**
```
{"key1": "value1", "key2": "value2"}
```

**Hint:** dumps() (with an 's', for 'string') returns a string — it doesn't write to a file.

In [ ]:
import json

data = {"key1": "value1", "key2": "value2"}

jsonData = json.dumps(data)
print(jsonData)
print(type(jsonData))

**Explanation:** json.dumps() ('dump string') converts a Python dictionary into its JSON text representation, returned as a plain Python str. The result looks visually similar to the original dict, but is now genuinely a string — type(jsonData) confirms this — ready to be sent over a network, written to a file, or embedded in an API response body.

## Exercise 2. Access a Value From a JSON String

**Concept:** json.loads()

**Problem:** Parse a JSON string and access the value associated with a specific key.

**Given:**
```
sampleJson = '{"key1": "value1", "key2": "value2"}'
```

**Expected Output:**
```
value2
```

**Hint:** loads() ('load string') is the reverse of dumps() — it parses JSON text back into a Python dict.

In [ ]:
import json

sampleJson = """{"key1": "value1", "key2": "value2"}"""

data = json.loads(sampleJson)
print(data['key2'])

**Explanation:** json.loads() parses the JSON text into an ordinary Python dictionary, after which data['key2'] is just standard dictionary key access — no different from working with any dict built directly in Python code.

## Exercise 3. Pretty-Print JSON With Custom Formatting

**Concept:** indent= and separators= in json.dumps()

**Problem:** Pretty-print a dictionary as JSON, with an indent level of 2 and custom key-value separators.

**Given:**
```
sampleJson = {"key1": "value1", "key2": "value2", "key3": "value3"}
```

**Expected Output:**
```
{
  "key1" = "value1",
  "key2" = "value2",
  "key3" = "value3"
}
```

**Hint:** separators is a 2-tuple: (between-items separator, key-to-value separator).

In [ ]:
import json

sampleJson = {"key1": "value1", "key2": "value2", "key3": "value3"}
prettyPrintedJson = json.dumps(sampleJson, indent=2, separators=(",", " = "))
print(prettyPrintedJson)

**Explanation:** indent=2 tells dumps() to spread the output across multiple lines, with 2 spaces of indentation per nesting level, instead of its default compact single-line output. separators=(",", " = ") is a 2-tuple controlling two specific punctuation choices: the first string separates consecutive items, and the second separates each key from its value — here producing a non-standard ' = ' instead of JSON's normal ': ', purely for this exercise's custom-formatting demonstration (note: output formatted this way with '=' instead of ':' is no longer valid, re-parseable JSON).

## Exercise 4. Sort JSON Keys and Write to a File

**Concept:** sort_keys=True combined with json.dump() (file version)

**Problem:** Write a dictionary to a JSON file, with its keys sorted alphabetically.

**Given:**
```
sampleJson = {"id": 1, "name": "value2", "age": 29}
```

**Expected Output:**
```
{
    "age": 29,
    "id": 1,
    "name": "value2"
}
```

**Hint:** json.dump() (no 's') writes DIRECTLY to a file object — you never need to create an intermediate string yourself.

In [ ]:
import json

sampleJson = {"id": 1, "name": "value2", "age": 29}

print("Started writing JSON data into a file")
with open("sampleJson.json", "w") as write_file:
    json.dump(sampleJson, write_file, indent=4, sort_keys=True)
print("Done writing JSON data into a file")

with open("sampleJson.json", "r") as f:
    print("\nFile contents:")
    print(f.read())

**Explanation:** json.dump() (no trailing 's') writes JSON text directly into an open file object, rather than json.dumps(), which would require you to manually write the returned string to a file yourself in a separate step. sort_keys=True reorders the output alphabetically by key ('age' before 'id' before 'name'), regardless of the order the keys were originally inserted into the dictionary.

## Exercise 5. Access a Deeply Nested Key

**Concept:** chained bracket access through multiple nesting levels

**Problem:** Access a value nested three levels deep inside a JSON structure.

**Given:**
```
sampleJson = {"company": {"employee": {"name": "emma", "payable": {"salary": 7000, "bonus": 800}}}}
```

**Expected Output:**
```
7000
```

**Hint:** Each bracket lookup descends exactly one level deeper into the parsed structure.

In [ ]:
import json

sampleJson = """{
   "company":{
      "employee":{
         "name":"emma",
         "payable":{
            "salary":7000,
            "bonus":800
         }
      }
   }
}"""

data = json.loads(sampleJson)
print(data['company']['employee']['payable']['salary'])

**Explanation:** After json.loads() parses the text, the result is just ordinary nested Python dictionaries — no special JSON-specific access syntax is needed. Each successive ['key'] lookup descends one level deeper: ['company'] reaches the outer dict, ['employee'] reaches the next dict inside it, ['payable'] reaches the one after that, and finally ['salary'] retrieves the target value at the bottom.

## Exercise 6. Convert a Custom Object Into JSON

**Concept:** JSONEncoder subclassing with a default() override

**Problem:** Convert an instance of a custom class into a JSON string.

**Given:**
```
vehicle = Vehicle("Toyota Rav4", "2.5L", 32000)
```

**Expected Output:**
```
{
    "name": "Toyota Rav4",
    "engine": "2.5L",
    "price": 32000
}
```

**Hint:** json.dumps() doesn't know how to serialize a custom class by default — it raises TypeError unless you tell it how.

In [ ]:
import json
from json import JSONEncoder

class Vehicle:
    def __init__(self, name, engine, price):
        self.name = name
        self.engine = engine
        self.price = price

class VehicleEncoder(JSONEncoder):
    def default(self, o):
        return o.__dict__

vehicle = Vehicle("Toyota Rav4", "2.5L", 32000)

print("Encode Vehicle Object into JSON")
vehicleJson = json.dumps(vehicle, indent=4, cls=VehicleEncoder)
print(vehicleJson)

# A simpler alternative for straightforward cases: just pass __dict__ directly
print("\nSimpler alternative:")
print(json.dumps(vehicle.__dict__, indent=4))

**Explanation:** json.dumps() only knows how to serialize Python's built-in types out of the box; passing it a custom class instance directly raises a TypeError, since it has no idea how to turn a Vehicle into JSON. VehicleEncoder subclasses JSONEncoder and overrides its .default() method — called specifically for any object dumps() doesn't already know how to handle — returning o.__dict__, the object's own attribute dictionary, which dumps() CAN serialize normally. cls=VehicleEncoder tells dumps() to use this custom encoder. For a case this simple, directly passing vehicle.__dict__ to a plain json.dumps() call achieves the identical result with less code — the custom encoder approach earns its complexity when the conversion logic needs to be more involved than a flat attribute dump.

## Exercise 7. Convert JSON Into a Custom Object

**Concept:** object_hook in json.loads()

**Problem:** Parse a JSON string directly into an instance of a custom class.

**Given:**
```
'{ "name": "Toyota Rav4", "engine": "2.5L", "price": 32000 }'
```

**Expected Output:**
```
Type: <class '__main__.Vehicle'>
Toyota Rav4 2.5L 32000
```

**Hint:** object_hook is called on every JSON OBJECT encountered during parsing, converting it from a plain dict into whatever your function returns.

In [ ]:
import json

class Vehicle:
    def __init__(self, name, engine, price):
        self.name = name
        self.engine = engine
        self.price = price

def vehicleDecoder(obj):
    return Vehicle(obj['name'], obj['engine'], obj['price'])

vehicleObj = json.loads('{ "name": "Toyota Rav4", "engine": "2.5L", "price": 32000 }',
                        object_hook=vehicleDecoder)

print("Type of decoded object from JSON Data")
print(type(vehicleObj))
print("Vehicle Details")
print(vehicleObj.name, vehicleObj.engine, vehicleObj.price)

**Explanation:** Without object_hook, json.loads() would parse this JSON into a plain dict, requiring bracket access like result['name']. Passing object_hook=vehicleDecoder tells the parser to call vehicleDecoder on every JSON object it encounters during parsing, using whatever that function returns in place of the plain dict — here, a genuine Vehicle instance. The result is that vehicleObj.name, vehicleObj.engine, and vehicleObj.price are all accessible directly via dot notation, exactly as requested, rather than dictionary-style bracket access.

## Exercise 8. Validate Whether a JSON String Is Valid

**Concept:** catching json.JSONDecodeError (a ValueError subclass)

**Problem:** Check whether a given JSON string is valid, and if not, identify and fix the syntax error.

**Given:**
```
InvalidJsonData with a missing comma between two key-value pairs
```

**Expected Output:**
```
Given JSON string is Valid False
Given JSON string is Valid True (after adding the missing comma)
```

**Hint:** json.JSONDecodeError IS a ValueError — catching ValueError catches this specific error too, plus any other value-parsing issue.

In [ ]:
import json

def validateJSON(jsonData):
    try:
        json.loads(jsonData)
    except ValueError:
        return False
    return True

InvalidJsonData = """{ "company":{ "employee":{ "name":"emma", "payble":{ "salary":7000 "bonus":800} } } }"""
isValid = validateJSON(InvalidJsonData)
print("Given JSON string is Valid", isValid)

# The fix: a comma is missing between "salary":7000 and "bonus":800
FixedJsonData = """{ "company":{ "employee":{ "name":"emma", "payble":{ "salary":7000, "bonus":800} } } }"""
isValidNow = validateJSON(FixedJsonData)
print("Given JSON string is Valid", isValidNow, "(after adding the missing comma)")

**Explanation:** json.loads() raises json.JSONDecodeError on malformed input — and since that specific exception class is actually a subclass of the more general ValueError, catching ValueError (as this validateJSON() function does) reliably catches it, along with any other value-related parsing issue. The specific bug in the invalid sample is a missing comma between "salary":7000 and "bonus":800 — two key-value pairs inside the same JSON object always need a comma between them, exactly like items in a Python dict literal.

## Exercise 9. Extract All Values of a Key From an Array of Objects

**Concept:** a list comprehension over parsed JSON array data

**Problem:** Parse a JSON array of objects and extract every value associated with the 'name' key.

**Given:**
```
a JSON array containing two objects, each with an id, name, and color list
```

**Expected Output:**
```
["name1", "name2"]
```

**Hint:** Once parsed, a JSON array of objects becomes a plain Python list of dicts — ordinary list/dict operations apply directly.

In [ ]:
import json

sampleJson = """[
   {
      "id":1,
      "name":"name1",
      "color":[
         "red",
         "green"
      ]
   },
   {
      "id":2,
      "name":"name2",
      "color":[
         "pink",
         "yellow"
      ]
   }
]"""

data = []
try:
    data = json.loads(sampleJson)
except Exception as e:
    print(e)

dataList = [item.get('name') for item in data]
print(dataList)

**Explanation:** After json.loads() parses the outer [...] array, the result is an ordinary Python list, with each of its elements being an ordinary dict (parsed from each {...} object). The comprehension [item.get('name') for item in data] loops through that list, pulling just the 'name' field out of each dict. .get('name') is used rather than item['name'] so that an object missing the 'name' key entirely would contribute None to the result instead of raising a KeyError and crashing the whole comprehension.